# SEACO-6161 · Table row counts and sizes

Lists every table in the chosen databases with its database name, type, format, **row count**,
**live size** and file count. Read only: it changes nothing and writes no audit table.
Standalone (no `%run`), so it works in any workspace (OPAS or ARC).

- `databases`: comma-separated list. Leave it empty to scan every database (slow on large workspaces).
- `skip_views`: counting a view runs its whole query, so views are skipped by default.
- `scan_files`: `true` counts rows by reading the data files instead of the Delta log statistics.
  Slower, but it proves the files are still readable (useful before and after a VACUUM).

**Size is the live size** (`DESCRIBE DETAIL` → `sizeInBytes`): the files the current version uses.
Old versions that VACUUM would remove are not included. For on-disk size and VACUUM overhead, use the
`table_inventory` table from `01_investigation`. Non-Delta tables may show no size or file count.

In [0]:
dbutils.widgets.text("databases", "", "1. Databases (comma list, empty = all)")
dbutils.widgets.dropdown("skip_views", "true", ["true", "false"], "2. Skip views")
dbutils.widgets.dropdown("scan_files", "false", ["true", "false"], "3. Count by reading data files")

_dbs = [d.strip() for d in dbutils.widgets.get("databases").split(",") if d.strip()]
DATABASES = _dbs or sorted(d.name for d in spark.catalog.listDatabases())
SKIP_VIEWS = dbutils.widgets.get("skip_views") == "true"
SCAN_FILES = dbutils.widgets.get("scan_files") == "true"

In [0]:
def fmt_size(b):
    if b is None:
        return "-"
    for unit in ["B", "KiB", "MiB", "GiB", "TiB"]:
        if b < 1024 or unit == "TiB":
            return f"{b:,.1f} {unit}"
        b /= 1024

META_KEY = "spark.databricks.delta.optimizeMetadataQuery.enabled"
_old_meta = spark.conf.get(META_KEY, "true")
if SCAN_FILES:
    spark.conf.set(META_KEY, "false")   # force count(*) to read the data files

results = []
try:
    for db in DATABASES:
        try:
            tables = spark.catalog.listTables(db)
        except Exception as e:
            results.append((db, None, None, None, None, None, None, f"list failed: {str(e)[:200]}"))
            continue
        for t in tables:
            if t.isTemporary:
                continue
            name = f"`{db}`.`{t.name}`"
            if t.tableType == "VIEW" and SKIP_VIEWS:
                results.append((db, t.name, t.tableType, None, None, None, None, "skipped (view)"))
                continue
            fmt = size = files = None
            try:
                if t.tableType != "VIEW":
                    d = spark.sql(f"DESCRIBE DETAIL {name}").first().asDict()
                    fmt, size, files = d.get("format"), d.get("sizeInBytes"), d.get("numFiles")
                cnt = spark.sql(f"SELECT count(*) AS c FROM {name}").first()["c"]
                results.append((db, t.name, t.tableType, fmt, cnt, size, files, None))
            except Exception as e:
                results.append((db, t.name, t.tableType, fmt, None, size, files, str(e)[:200]))
finally:
    spark.conf.set(META_KEY, _old_meta)

counts_df = spark.createDataFrame(
    results,
    "database string, table string, type string, format string, row_count long, "
    "size_bytes long, num_files long, error string",
).orderBy("database", "table")

In [0]:
print(f"{'database':32s} {'table':50s} {'rows':>15s} {'size':>12s} {'files':>10s}")
for db, tbl, ttype, fmt, cnt, size, files, err in sorted(results, key=lambda r: (r[0], r[1] or "")):
    rows = f"{cnt:,}" if cnt is not None else "-"
    nf = f"{files:,}" if files is not None else "-"
    print(f"{db:32s} {(tbl or '-'):50s} {rows:>15s} {fmt_size(size):>12s} {nf:>10s}  {err or ''}")

print()
for db in DATABASES:
    sub = [r for r in results if r[0] == db and r[1] is not None]
    print(f"{db}: {len(sub)} tables, {sum(r[4] or 0 for r in sub):,} rows, "
          f"{fmt_size(sum(r[5] or 0 for r in sub))} live, {sum(1 for r in sub if r[7] and r[7] != 'skipped (view)')} errors")

In [0]:
from pyspark.sql import functions as F

display(counts_df.withColumn("size_gib", (F.col("size_bytes") / 1024**3).cast("decimal(14,2)")))

In [0]:
display(
    counts_df.groupBy("database")
    .agg(F.count("table").alias("tables"),
         F.sum("row_count").alias("rows"),
         (F.sum("size_bytes") / 1024**3).cast("decimal(14,2)").alias("size_gib"),
         F.sum("num_files").alias("files"),
         F.count("error").alias("errors_or_skipped"))
    .orderBy(F.desc("size_gib"))
)